# **Revisi Generator Variasi Bahasa Lokal — Development v5**

---

**Proyek:** KP Agentic AI + LLM  
**Tahap:** Step 14 — Perbaikan Variasi Berdasarkan Audit  
**Dataset:** MASSIVE-Agents `id-ID` development

Revisi v5 dilakukan berdasarkan hasil audit 971 kandidat v4. Fokusnya adalah memperbaiki variasi yang terindikasi bermasalah dan memisahkan kasus yang belum dapat dipastikan. GT dan pembagian development tetap mengacu pada data sumber, sedangkan hasil v4 dan pilot sebelumnya dipertahankan.


## **1. Dataset dan Konfigurasi**

---

Data yang digunakan meliputi 971 kandidat v4, hasil audit Step 14, dan 397 instruksi Indonesia pada development set. Pemeriksaan dilakukan menggunakan data dan konfigurasi yang sama agar perubahan dari v4 ke v5 dapat ditelusuri.


In [1]:
from __future__ import annotations
import csv, hashlib, json, os, re
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

SEED = 20261110
ROOT_OVERRIDE = None
CATEGORIES = ["slang", "singkatan", "typo", "code_switch_id_en", "code_switch_id_jawa"]


def cari_root():
    candidates = []
    if ROOT_OVERRIDE:
        candidates.append(Path(ROOT_OVERRIDE))
    if os.environ.get("KP_AGENTIC_AI_ROOT"):
        candidates.append(Path(os.environ["KP_AGENTIC_AI_ROOT"]))
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents, Path(r"D:\KP-Agentic-AI")])
    for root in candidates:
        d = root / 'results' / 'variasi_bahasa_terkontrol' / 'development_v4'
        if (d/'kandidat_variasi_development_v4.csv').is_file() and (d/'STEP14_AUDIT_971_MESIN.csv').is_file() and (root/'data'/'processed'/'massive_agents_reviewed'/'id-ID_development.jsonl').is_file():
            return root
    raise FileNotFoundError("Sumber data V4, audit Step 14, atau development JSONL tidak ditemukan pada lokasi proyek.")

ROOT = cari_root()
INPUT_V4 = ROOT/'results'/'variasi_bahasa_terkontrol'/'development_v4'/'kandidat_variasi_development_v4.csv'
INPUT_AUDIT = ROOT/'results'/'variasi_bahasa_terkontrol'/'development_v4'/'STEP14_AUDIT_971_MESIN.csv'
INPUT_SOURCE = ROOT/'data'/'processed'/'massive_agents_reviewed'/'id-ID_development.jsonl'
OUTPUT = ROOT/'results'/'variasi_bahasa_terkontrol'/'development_v5'


def read_csv(path):
    with path.open('r',encoding='utf-8-sig',newline='') as f:
        return list(csv.DictReader(f))


def hash_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

hash_v4_awal = hash_file(INPUT_V4)
print('Project:', ROOT)
print('Input V4:', INPUT_V4)
print('Audit Step 14:', INPUT_AUDIT)
print('Output V5:', OUTPUT)


Project: D:\KP-Agentic-AI
Input V4: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v4\kandidat_variasi_development_v4.csv
Audit Step 14: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v4\STEP14_AUDIT_971_MESIN.csv
Output V5: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v5


## **2. Pemeriksaan Sumber Data**

---

Kandidat v4 dan audit dicocokkan berdasarkan `variant_id`, kategori, instruksi induk, serta GT. Kesesuaian dengan data development turut diperiksa untuk memastikan bahwa revisi tidak menggunakan test set atau mengubah label sumber.


In [2]:
v4 = read_csv(INPUT_V4)
audit_rows = read_csv(INPUT_AUDIT)
with INPUT_SOURCE.open('r',encoding='utf-8') as f:
    development = [json.loads(line) for line in f if line.strip()]

sumber = {r['row_key']: r for r in development}
audit_by_id = {r['variant_id']:r for r in audit_rows}
assert len(v4) == 971 and len(development) == 397 and len(audit_rows) == 971
assert len(audit_by_id) == len(audit_rows)
assert len({r['variant_id'] for r in v4}) == len(v4)
assert {r['variant_id'] for r in v4} == set(audit_by_id), 'Audit dan V4 tidak identik'

for r in v4:
    a = audit_by_id[r['variant_id']]
    parent = sumber[r['row_key_induk']]
    assert a['row_key_induk'] == r['row_key_induk']
    assert a['kategori'] == r['kategori']
    assert r['split'] == parent['split'] == 'development'
    assert json.loads(r['gt_json']) == parent['gt']
    assert r['instruksi_id_asli'] == parent['question']

print('Data sumber:',len(development))
print('Kandidat V4:',len(v4))
print('Audit Step 14:',len(audit_rows))
print('Status audit AI:',dict(Counter(r['status_tinjauan_ai'] for r in audit_rows)))
print('Induk terindikasi perlu cek GT:',len({r['row_key_induk'] for r in audit_rows if r['prioritas_gt_sumber']=='YA'}))


Data sumber: 397
Kandidat V4: 971
Audit Step 14: 971
Status audit AI: {'BELUM_REVIEW_SEMANTIK': 280, 'LOLOS_TINJAUAN_AI': 387, 'CEK_PENUTUR_JAWA': 186, 'CEK_GT_SUMBER': 43, 'CEK_KEWAJARAN_KONTEKS': 42, 'REVISI_VARIASI': 33}
Induk terindikasi perlu cek GT: 19


## **3. Hasil Audit Variasi v4**

---

Hasil audit dikelompokkan menurut kategori dan jenis temuan. Penilaian AI digunakan sebagai dasar pemilihan kasus revisi, tetapi tidak dianggap sebagai bukti validasi semantik oleh manusia.


In [3]:
ringkasan_audit = Counter(row["status_tinjauan_ai"] for row in audit_rows)
ringkasan_kategori = Counter(row["kategori"] for row in v4)

print("Kandidat V4 per kategori:")
for kategori in CATEGORIES:
    print(f"  {kategori}: {ringkasan_kategori[kategori]}")

print("\nTemuan tinjauan AI:")
for status, jumlah in ringkasan_audit.most_common():
    print(f"  {status}: {jumlah}")

print("\nCatatan: tinjauan AI belum menggantikan validasi manusia.")


Kandidat V4 per kategori:
  slang: 139
  singkatan: 187
  typo: 304
  code_switch_id_en: 128
  code_switch_id_jawa: 213

Temuan tinjauan AI:
  LOLOS_TINJAUAN_AI: 387
  BELUM_REVIEW_SEMANTIK: 280
  CEK_PENUTUR_JAWA: 186
  CEK_GT_SUMBER: 43
  CEK_KEWAJARAN_KONTEKS: 42
  REVISI_VARIASI: 33

Catatan: tinjauan AI belum menggantikan validasi manusia.


## **4. Pemeriksaan Nilai Argumen GT**

---

Nilai argumen GT yang tertulis dalam instruksi induk dijadikan acuan ketika memeriksa usulan perubahan teks. Variasi yang menghilangkan nilai eksplisit tersebut tidak diterima. Pemeriksaan ini terbatas pada kecocokan teks dan belum mencakup seluruh kemungkinan perubahan makna.


In [4]:
def iter_teks_gt(data):
    if isinstance(data,dict):
        for v in data.values(): yield from iter_teks_gt(v)
    elif isinstance(data,(tuple,list)):
        for v in data: yield from iter_teks_gt(v)
    elif isinstance(data,str) and data.strip():
        yield data.strip()


def nilai_gt_terjaga(original, updated, gt):
    for value in iter_teks_gt(gt):
        if value.casefold() in {'now','current','null','none'}: continue
        pattern = r'(?<!\w)' + re.escape(value) + r'(?!\w)'
        if re.search(pattern,original,re.I) and not re.search(pattern,updated,re.I):
            return False, value
    return True, ''


## **5. Perbaikan Aturan Variasi**

---

Perbaikan difokuskan pada temuan yang memiliki pola cukup jelas, terutama ketidakkonsistenan pronomina dalam kategori slang dan beberapa perubahan frasa ID–EN. Kasus yang dapat mengubah tindakan atau konteks tetap dipisahkan untuk ditinjau lebih lanjut. Variasi ID–Jawa juga masih memerlukan pemeriksaan penutur.


In [5]:
def perbaiki_variasi(row):
    """Kembalikan (teks, alasan). Tidak ada fallback yang diam-diam dianggap valid."""
    original = row['instruksi_id_asli']
    category = row['kategori']
    gt_function = row['fungsi_gt']
    if category == 'slang' and re.search(r'\bsaya\b', original, re.I):
        # Ganti SEMUA pronomina terkait agar tidak ada campuran "aku" dan "saya".
        t = re.sub(r'\bsaya\b','gue',original,flags=re.I)
        # Perbaikan leksikal terbatas pada konteks yang sudah muncul dalam audit.
        t = re.sub(r'\bdapatkan gue\b','kasih gue',t,flags=re.I)
        t = re.sub(r'\bmainkan gue (.+)$',r'ayo main \1 sama gue',t,flags=re.I)
        t = re.sub(r'\btampilkan gue\b','tunjukin gue',t,flags=re.I)
        t = re.sub(r'\bbacakan gue\b','bacain gue',t,flags=re.I)
        t = re.sub(r'\bbiarkan gue\b','biarin gue',t,flags=re.I)
        t = re.sub(r'\bingatkan gue\b','ingetin gue',t,flags=re.I)
        return t,'Ganti seluruh pronomina saya→gue; penyelarasan ragam slang'
    if category == 'code_switch_id_en':
        m = re.match(r'^bacakan saya (.+)$',original,re.I)
        if m:
            obj = m.group(1)
            # Jika objek sudah memuat 'saya' (misalnya 'feed saya'),
            # tidak perlu menambahkan 'untuk saya' di akhir.
            if re.search(r'\bsaya\b',obj,re.I):
                return f'tolong read aloud {obj}','Read aloud + objek, rujukan kepemilikan tetap'
            return f'tolong read aloud {obj} untuk saya','Read aloud + objek + penerima'
        if re.match(r'^carikan saya lelucon\b',original,re.I):
            return re.sub(r'\blelucon\b','joke',original,count=1,flags=re.I),'Code-switch kata benda lelucon→joke; tidak menghapus penerima'
        if gt_function == 'play.game' and re.match(r'^mainkan saya (.+)$',original,re.I):
            game = re.match(r'^mainkan saya (.+)$',original,re.I).group(1)
            return f'play game {game} sama saya','Konteks permainan dipertahankan, bukan memutar media'
        # Bahasa Inggris generik lain sengaja tidak ditebak.
    if category == 'code_switch_id_jawa':
        m = re.match(r'^bangunkan saya (.+)$',original,re.I)
        if m:
            s = re.sub(r'\bsaya\b','aku',m.group(1),flags=re.I)
            return 'tulung gugah aku '+s,'Usulan Jawa ngoko: tulung gugah aku (tetap perlu penutur)'
    return None, 'Belum ada aturan revisi otomatis yang cukup aman; ditahan untuk review'

print('Aturan revisi konservatif siap. Tidak ada perubahan pada GT.')


Aturan revisi konservatif siap. Tidak ada perubahan pada GT.


## **6. Pembentukan Kandidat v5**

---

Setiap kandidat v4 dipertahankan keterkaitannya dengan instruksi induk dan hasil audit. Teks hanya diperbarui ketika terdapat aturan revisi yang sesuai. Kandidat lain tetap dicatat dengan status tinjauan masing-masing agar perubahan dan kasus yang tertunda dapat dibedakan.


In [6]:
hasil = []
log_perubahan = []

for source in v4:
    row = dict(source)
    aid = row['variant_id']
    a = audit_by_id[aid]
    old = row['instruksi_variasi']
    row['instruksi_variasi_v4'] = old
    row['status_audit_step14'] = a['status_tinjauan_ai']
    row['temuan_audit_step14'] = a['temuan_tinjauan_ai']
    row['asal_tinjauan'] = a['sumber_tinjauan']
    row['gt_sumber_perlu_cek'] = a['prioritas_gt_sumber']
    row['validasi_manusia'] = 'BELUM'
    row['revisi_v5'] = 'TIDAK'
    row['catatan_revisi_v5'] = ''
    row['versi_generator'] = 'v5_audit_aware'
    row['status_validasi'] = 'BELUM_VALIDASI_MANUSIA'
    row['approved_for_final_model_evaluation'] = 'TIDAK'

    # Perbaiki hanya kasus yang diperiksa dan ditandai "REVISI_VARIASI".
    if a['status_tinjauan_ai'] == 'REVISI_VARIASI':
        proposal, note = perbaiki_variasi(row)
        if proposal is not None and proposal != old and proposal != row['instruksi_id_asli']:
            ok, violated = nilai_gt_terjaga(row['instruksi_id_asli'], proposal, json.loads(row['gt_json']))
            if ok:
                row['instruksi_variasi'] = proposal
                row['revisi_v5'] = 'YA'
                row['catatan_revisi_v5'] = note
            else:
                row['catatan_revisi_v5'] = 'DITAHAN: nilai GT hilang setelah revisi: '+violated
        else:
            row['catatan_revisi_v5'] = note

    # Prioritas dipilih secara konservatif: risiko GT > Jawa > perubahan konteks.
    if row['gt_sumber_perlu_cek'] == 'YA' or a['status_tinjauan_ai'] == 'CEK_GT_SUMBER':
        row['keputusan_v5'] = 'TAHAN_CEK_GT_SUMBER'
    elif row['kategori'] == 'code_switch_id_jawa':
        row['keputusan_v5'] = 'TAHAN_CEK_PENUTUR_JAWA'
    elif a['status_tinjauan_ai'] == 'REVISI_VARIASI' and row['revisi_v5'] == 'TIDAK':
        row['keputusan_v5'] = 'TAHAN_REVISI_MANUAL'
    elif row['revisi_v5'] == 'YA':
        row['keputusan_v5'] = 'REVIEW_MANUSIA_SETELAH_REVISI'
    elif a['status_tinjauan_ai'] == 'CEK_KEWAJARAN_KONTEKS':
        row['keputusan_v5'] = 'TAHAN_CEK_KONTEKS'
    elif a['status_tinjauan_ai'] == 'BELUM_REVIEW_SEMANTIK':
        row['keputusan_v5'] = 'BELUM_REVIEW_SEMANTIK'
    else:
        row['keputusan_v5'] = 'LOLOS_TINJAUAN_AI_BELUM_HUMAN'

    if row['revisi_v5'] == 'YA' or row['keputusan_v5'].startswith('TAHAN_'):
        log_perubahan.append({
            'variant_id':aid, 'kategori':row['kategori'],
            'status_audit_step14':row['status_audit_step14'],
            'keputusan_v5':row['keputusan_v5'],
            'instruksi_asli':row['instruksi_id_asli'],
            'instruksi_v4':old, 'instruksi_v5':row['instruksi_variasi'],
            'teks_berubah':row['revisi_v5'], 'alasan':row['catatan_revisi_v5'] or row['temuan_audit_step14'],
            'gt_sumber_perlu_cek':row['gt_sumber_perlu_cek'],
        })
    hasil.append(row)

assert len(hasil) == len(v4) == 971
print('Usulan teks direvisi:',sum(r['revisi_v5']=='YA' for r in hasil))
print('Belum diperbaiki otomatis:',sum(r['status_audit_step14']=='REVISI_VARIASI' and r['revisi_v5']=='TIDAK' for r in hasil))
print('Status keputusan V5:')
for k,n in Counter(r['keputusan_v5'] for r in hasil).most_common(): print(f'  {k}: {n}')


Usulan teks direvisi: 20
Belum diperbaiki otomatis: 13
Status keputusan V5:
  LOLOS_TINJAUAN_AI_BELUM_HUMAN: 384
  BELUM_REVIEW_SEMANTIK: 271
  TAHAN_CEK_PENUTUR_JAWA: 199
  TAHAN_CEK_GT_SUMBER: 60
  TAHAN_CEK_KONTEKS: 40
  REVIEW_MANUSIA_SETELAH_REVISI: 15
  TAHAN_REVISI_MANUAL: 2


## **7. Pemeriksaan Konsistensi**

---

Pemeriksaan dilakukan terhadap `group_id`, split, fungsi, GT, instruksi induk, dan nilai argumen yang tampak dalam teks. Hasil pemeriksaan struktur dipisahkan dari keputusan validasi semantik karena keduanya mengukur hal yang berbeda.


In [7]:
cek = []
for row in hasil:
    parent = sumber[row['row_key_induk']]
    original_gt = parent['gt']
    preserved, missing_val = nilai_gt_terjaga(parent['question'],row['instruksi_variasi'],original_gt)
    status = {
        'variant_id':row['variant_id'],
        'gt_tidak_diubah':json.loads(row['gt_json']) == original_gt,
        'fungsi_gt_sama':row['fungsi_gt'] == next(iter(original_gt)),
        'group_id_sama':row['group_id'] == parent['group_id'],
        'development_only':row['split'] == parent['split'] == 'development',
        'instruksi_induk_sama':row['instruksi_id_asli'] == parent['question'],
        'variasi_berbeda_dari_asli':row['instruksi_variasi'] != parent['question'],
        'nilai_slot_eksplisit_terjaga':preserved,
        'nilai_slot_hilang':missing_val,
    }
    cek.append(status)
assert len({r['variant_id'] for r in hasil}) == 971
assert all(all(v is True for k,v in c.items() if k not in ('variant_id','nilai_slot_hilang')) for c in cek), 'Ada pelanggaran pemeriksaan mekanis.'
assert all(r['approved_for_final_model_evaluation']=='TIDAK' for r in hasil)
assert hash_file(INPUT_V4) == hash_v4_awal,'File V4 berubah; hentikan proses!'
print('Pemeriksaan mekanis 971/971 PASS')
print('Status semantik: BELUM VALIDASI MANUSIA')
print('Ground truth sumber: TIDAK diubah')
print('File V4: tetap identik')


Pemeriksaan mekanis 971/971 PASS
Status semantik: BELUM VALIDASI MANUSIA
Ground truth sumber: TIDAK diubah
File V4: tetap identik


## **8. Ringkasan Hasil dan Kasus Prioritas**

---

Perubahan teks dihitung untuk setiap kategori dan dikelompokkan bersama kasus yang masih memerlukan pemeriksaan GT, konteks, atau kewajaran bahasa. Pemisahan ini digunakan untuk melihat jenis masalah yang masih tersisa setelah revisi otomatis.


In [8]:
status_prioritas={'TAHAN_CEK_GT_SUMBER','TAHAN_CEK_PENUTUR_JAWA','TAHAN_REVISI_MANUAL','TAHAN_CEK_KONTEKS','REVIEW_MANUSIA_SETELAH_REVISI','BELUM_REVIEW_SEMANTIK'}
prioritas=[r for r in hasil if r['keputusan_v5'] in status_prioritas]
ringkasan=[]
for category in CATEGORIES:
    r=[x for x in hasil if x['kategori']==category]
    ringkasan.append({
        'kategori':category, 'total_v4':len(r), 'total_v5':len(r),
        'teks_direvisi':sum(x['revisi_v5']=='YA' for x in r),
        'prioritas_gt':sum(x['gt_sumber_perlu_cek']=='YA' for x in r),
        'lolos_tinjauan_ai_belum_human':sum(x['keputusan_v5']=='LOLOS_TINJAUAN_AI_BELUM_HUMAN' for x in r),
        'boleh_evaluasi_final':0,
    })


print("Ringkasan hasil V5:")
for item in ringkasan:
    print(f"  {item['kategori']}: {item['total_v5']} kandidat; {item['teks_direvisi']} usulan revisi")
print("Kasus pada daftar prioritas:", len(prioritas))


Ringkasan hasil V5:
  slang: 139 kandidat; 12 usulan revisi
  singkatan: 187 kandidat; 0 usulan revisi
  typo: 304 kandidat; 0 usulan revisi
  code_switch_id_en: 128 kandidat; 4 usulan revisi
  code_switch_id_jawa: 213 kandidat; 4 usulan revisi
Kasus pada daftar prioritas: 587


## **9. Dokumentasi Hasil v5**

---

Hasil v5 mencakup kandidat hasil revisi, daftar perubahan teks, pemeriksaan konsistensi, ringkasan kategori, serta metadata proses. Setiap hasil ditempatkan pada versi tersendiri sehingga data v4 tetap dapat digunakan sebagai pembanding.


In [9]:
OUTPUT.mkdir(parents=True,exist_ok=True)


def save_csv(rows,name):
    if not rows: return None
    keys=list(dict.fromkeys(k for item in rows for k in item))
    with (OUTPUT/name).open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=keys);writer.writeheader();writer.writerows(rows)
    return OUTPUT/name


paths = [
    save_csv(hasil,'kandidat_variasi_development_v5.csv'),
    save_csv(log_perubahan,'perubahan_dari_v4_ke_v5.csv'),
    save_csv(prioritas,'prioritas_review_v5.csv'),
    save_csv(cek,'pemeriksaan_konsistensi_v5.csv'),
    save_csv(ringkasan,'ringkasan_v4_vs_v5.csv'),
]
meta={
    'run':'generator_v5_audit_step14', 'created_utc':datetime.now(timezone.utc).isoformat(),
    'seed':SEED,'source_v4_sha256':hash_v4_awal,'audit_step14_sha256':hash_file(INPUT_AUDIT),
    'source_development_sha256':hash_file(INPUT_SOURCE),'source_development_rows':len(development),
    'variants_v4':len(v4),'variants_v5':len(hasil),
    'revised_texts':sum(x['revisi_v5']=='YA' for x in hasil),
    'structural_validation':'PASS','semantic_human_validation':'PENDING',
    'gt_modified':False,'test_used':False,'pilot_predictions_modified':False,
    'approved_for_final_model_evaluation':False,
    'counts':dict(Counter(x['kategori'] for x in hasil)),
    'status_counts':dict(Counter(x['keputusan_v5'] for x in hasil)),
    'note':'Hasil tinjauan AI tidak dapat menggantikan anotasi manusia atau membuktikan GT sumber benar.'
}
meta_file=OUTPUT/'metadata_development_v5.json'
meta_file.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
paths.append(meta_file)
assert hash_file(INPUT_V4)==hash_v4_awal
for p in paths:
    if p: print(p)
print('\nV5 dibuat. Belum diizinkan untuk evaluasi final tanpa audit manusia.')


D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v5\kandidat_variasi_development_v5.csv
D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v5\perubahan_dari_v4_ke_v5.csv
D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v5\prioritas_review_v5.csv
D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v5\pemeriksaan_konsistensi_v5.csv
D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v5\ringkasan_v4_vs_v5.csv
D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v5\metadata_development_v5.json

V5 dibuat. Belum diizinkan untuk evaluasi final tanpa audit manusia.


## **10. Kesimpulan Sementara**

---

Revisi v5 mempertahankan 971 kandidat agar seluruh perubahan dari v4 tetap dapat ditelusuri. Perbaikan otomatis hanya diterapkan pada pola yang cukup teridentifikasi, sementara kasus lain tetap diberi status pemeriksaan lanjutan. GT, split development, dan hasil pilot sebelumnya tidak diubah.

Lolosnya pemeriksaan struktur belum menunjukkan bahwa seluruh variasi setara secara makna. Kasus GT sumber, variasi yang direvisi, dan terutama variasi ID–Jawa masih memerlukan validasi lanjutan sebelum digunakan dalam evaluasi final.
